In [6]:
import numpy as np
from numba import cuda
import math

@cuda.jit
def square_pattern_kernel(d_grid, N):
    # Calculate 2D global thread coordinates
    x, y = cuda.grid(2)

    # Boundary check to ensure we stay within the square dimensions
    if x < N and y < N:
        # Check if the thread is on any of the 4 borders of the square
        if x == 0 or x == N - 1 or y == 0 or y == N - 1:
            d_grid[x, y] = 42  # ASCII code for '*'
        else:
            d_grid[x, y] = 32  # ASCII code for ' ' (space)

def generate_parallel_square(N):
    # Create an empty 2D grid array on the host
    h_grid = np.zeros((N, N), dtype=np.int32)

    # Allocate and copy grid to the GPU device
    d_grid = cuda.to_device(h_grid)

    # Configure 2D thread block layout (e.g., 16x16 threads per block)
    threads_per_block = (16, 16)

    # Calculate blocks needed for both X and Y dimensions
    blocks_x = math.ceil(N / threads_per_block[0])
    blocks_y = math.ceil(N / threads_per_block[1])
    blocks_per_grid = (blocks_x, blocks_y)

    # Launch the 2D parallel kernel
    square_pattern_kernel[blocks_per_grid, threads_per_block](d_grid, N)

    # Copy the finished matrix back to the host CPU
    return d_grid.copy_to_host()

if __name__ == "__main__":
    # Define the dimension of the square
    square_size = 15

    print(f"Generating a {square_size}x{square_size} hollow square pattern in parallel on the GPU...")
    print("-" * 60)

    # Run the GPU parallel generator
    gpu_matrix = generate_parallel_square(square_size)

    # Convert ASCII integers back to characters and print the row matrix
    for row in gpu_matrix:
        line = "".join(chr(ascii_val) for ascii_val in row)
        # Add a space between characters so the square looks geometrically even on screen
        print(" ".join(line))

    print("-" * 60)
    print("Success! Every coordinate was computed in parallel.")

Generating a 15x15 hollow square pattern in parallel on the GPU...
------------------------------------------------------------
* * * * * * * * * * * * * * *
*                           *
*                           *
*                           *
*                           *
*                           *
*                           *
*                           *
*                           *
*                           *
*                           *
*                           *
*                           *
*                           *
* * * * * * * * * * * * * * *
------------------------------------------------------------
Success! Every coordinate was computed in parallel.


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:748: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
